# ✈️ Flight Delay Prediction: Data Exploration & Schema Inspection
**Phase 1: Traditional Machine Learning Only**  
*Project Title: Flight Delay Prediction via Gradient Boosting and Temporal Deep Learning*

---

## 🎯 1. Project Objective & Scope
The objective of this project is to predict **whether a commercial flight will arrive at least 15 minutes late** (`Arrival Delay >= 15 minutes`) using **only information available prior to its scheduled departure**.

### ⚠️ Strict Anti-Leakage Rule
In flight operations, any information that occurs **after scheduled departure** (such as actual departure time, actual departure delay, taxi-out time, wheels-off time, airborne time, or actual arrival time) is strictly **forbidden as an input feature**. Using them would result in **data leakage**, rendering the model useless for real-world advance prediction.

---

## 📋 2. Notebook Structure & Tasks
1. **Environment Setup & Imports**: Load libraries and configure visual themes.
2. **Dataset Discovery & Path Resolution**: Automatically locate dataset files in `data/raw/` or `data/`.
3. **Column Mapping Verification**: Map raw column names to canonical flight concepts:
   - Scheduled Departure, Scheduled Arrival
   - Actual Departure, Actual Arrival (Leakage)
   - Airline, Origin, Destination, Distance
   - Cancellation, Diversion, Arrival Delay (Target)
4. **Data Leakage Audit**: Separate columns into *Safe Predictive Features*, *Target*, and *Forbidden Leakage*.
5. **Data Quality Assessment**: Row counts, data types, missing values, duplicates, and date range.
6. **Operational Distributions**: Analyze cancellations, diversions, and target class balance (${\ge} 15$ min delay).


### Step 1: Import Libraries and Setup Styling
We load standard data science libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`) and configure our plotting styles.

In [1]:
# Standard library imports
import os
import sys
from pathlib import Path

# Add project root to sys.path so src modules can be imported
PROJECT_ROOT = Path(os.path.abspath(".."))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Data manipulation and scientific computing
import numpy as np
import pandas as pd

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Project configuration and utilities
from src.config import (
    DATA_DIR,
    RAW_DATA_DIR,
    FIGURES_DIR,
    COLUMN_ALIASES,
    LEAKAGE_COLUMNS,
    TARGET_COLUMN,
    DELAY_THRESHOLD_MINUTES,
    RANDOM_STATE
)
from src.data_loader import (
    find_dataset_files,
    detect_column_mapping,
    audit_feature_leakage,
    inspect_dataset
)

# Plotting aesthetics configuration
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

print("✓ All libraries imported successfully.")
print(f"✓ Project Root: {PROJECT_ROOT}")


✓ All libraries imported successfully.
✓ Project Root: /Users/suryansh/Desktop/video/Flight_Delay_Prediction


### Step 2: Discover Available Datasets in Workspace
We scan `data/` and `data/raw/` for supported dataset files (`.csv`, `.parquet`, `.zip`).

In [2]:
# Search for dataset files
dataset_files = find_dataset_files()

print(f"Found {len(dataset_files)} dataset file(s):")
if not dataset_files:
    print("\n⚠️  No dataset files found in data/ or data/raw/.")
    print("Please place your flight delay dataset CSV (e.g. flights.csv) in data/raw/.")
    print("Refer to data/README.md for instructions.")
else:
    for idx, f in enumerate(dataset_files, 1):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  [{idx}] {f.name} ({size_mb:.2f} MB) -> {f}")


Found 0 dataset file(s):

⚠️  No dataset files found in data/ or data/raw/.
Please place your flight delay dataset CSV (e.g. flights.csv) in data/raw/.
Refer to data/README.md for instructions.


### Step 3: Load the Dataset
If a dataset is present, we load it into pandas. For very large datasets (> 100 MB), we can inspect an initial chunk (e.g., 100,000 rows) or load the full dataset depending on available memory.

*Note: If no dataset is placed yet, this notebook provides a demonstration schema checklist.*

In [3]:
# Determine primary dataset path
DATASET_AVAILABLE = len(dataset_files) > 0

if DATASET_AVAILABLE:
    primary_file = dataset_files[0]
    file_size_mb = primary_file.stat().st_size / (1024 * 1024)
    print(f"Inspecting primary dataset: {primary_file.name} ({file_size_mb:.1f} MB)")
    
    # For large files (> 100MB), sample initial 100k rows for rapid interactive exploration
    SAMPLE_ROWS = 100_000 if file_size_mb > 100 else None
    
    # Load dataset
    df = pd.read_csv(primary_file, nrows=SAMPLE_ROWS, low_memory=False)
    print(f"✓ Loaded {len(df):,} rows and {len(df.columns)} columns.")
else:
    print("⚠️  DATASET ABSENT: Please place your CSV file into 'data/raw/' to execute live data inspection.")
    print("Showing empty container for illustration.")
    df = pd.DataFrame()


⚠️  DATASET ABSENT: Please place your CSV file into 'data/raw/' to execute live data inspection.
Showing empty container for illustration.


### Step 4: Canonical Column Mapping & Availability Verification
We determine whether the dataset contains all required flight performance concepts:
1. Scheduled Departure
2. Scheduled Arrival
3. Actual Departure (Leakage)
4. Actual Arrival (Leakage)
5. Cancellation Flag
6. Diversion Flag
7. Airline / Carrier
8. Origin Airport
9. Destination Airport
10. Distance
11. Arrival Delay (Target source)

In [4]:
if not df.empty:
    detected_mapping, missing_concepts = detect_column_mapping(df.columns)
    
    # Create inspection summary dataframe
    checklist = [
        ("Scheduled Departure", "scheduled_departure", "Pre-departure Predictive Feature"),
        ("Scheduled Arrival", "scheduled_arrival", "Pre-departure Predictive Feature"),
        ("Actual Departure", "actual_departure", "⚠️ LEAKAGE (Excluded from model)"),
        ("Actual Arrival", "actual_arrival", "⚠️ LEAKAGE (Excluded from model)"),
        ("Arrival Delay", "arrival_delay", "🎯 TARGET VARIABLE SOURCE"),
        ("Airline / Carrier", "airline", "Pre-departure Predictive Feature"),
        ("Origin Airport", "origin", "Pre-departure Predictive Feature"),
        ("Destination Airport", "destination", "Pre-departure Predictive Feature"),
        ("Flight Distance", "distance", "Pre-departure Predictive Feature"),
        ("Cancellation Flag", "cancellation", "Operational Filter"),
        ("Diversion Flag", "diversion", "Operational Filter"),
    ]
    
    rows = []
    for label, concept, role in checklist:
        found_col = detected_mapping.get(concept, None)
        status = "✓ FOUND" if found_col else "❌ MISSING"
        rows.append({
            "Concept": label,
            "Detected Column in CSV": found_col if found_col else "None",
            "Status": status,
            "Role in Pipeline": role
        })
    
    mapping_df = pd.DataFrame(rows)
    print(mapping_df.to_string(index=False))
else:
    print("Awaiting dataset placement to perform schema mapping.")


Awaiting dataset placement to perform schema mapping.


### Step 5: Data Leakage Guardrail Audit
Here we explicitly audit every column in the dataset to partition them into:
1. **Safe Predictive Features**: Known strictly prior to scheduled departure.
2. **Target Candidates**: The actual arrival delay and binary delayed flag.
3. **Leakage Columns**: Any post-departure variable (actual departure time, taxi, wheels-off, delay breakdown causes).

In [5]:
if not df.empty:
    leakage_cols, safe_cols = audit_feature_leakage(df.columns)
    
    print("=" * 70)
    print("DATA LEAKAGE PREVENTION AUDIT")
    print("=" * 70)
    print(f"Total Columns Audited: {len(df.columns)}")
    print(f"✓ Safe Pre-Departure Columns  : {len(safe_cols)}")
    print(f"⚠️ Forbidden Leakage Columns   : {len(leakage_cols)}")
    print("-" * 70)
    print("Forbidden Leakage Columns detected:")
    for col in leakage_cols:
        print(f"  [EXCLUDE] -> {col}")
    print("=" * 70)
else:
    print("Awaiting dataset placement.")


Awaiting dataset placement.


### Step 6: Missing Values and Data Types Analysis
We examine each column's data type, number of null values, and null percentage. High missing rates in delay reason columns (e.g., `WEATHER_DELAY`) are normal in BTS data because they are only populated when a delay occurs.

In [6]:
if not df.empty:
    missing_summary = pd.DataFrame({
        "Column": df.columns,
        "Dtype": [str(df[col].dtype) for col in df.columns],
        "Missing_Count": df.isnull().sum().values,
        "Missing_Percentage": (df.isnull().sum().values / len(df)) * 100
    }).sort_values(by="Missing_Percentage", ascending=False).reset_index(drop=True)
    
    print("Top 15 Columns by Missing Percentage:")
    print(missing_summary.head(15).to_string(index=False))
    
    # Plot missing values
    plt.figure(figsize=(12, 6))
    top_missing = missing_summary[missing_summary["Missing_Percentage"] > 0].head(20)
    if not top_missing.empty:
        sns.barplot(data=top_missing, x="Missing_Percentage", y="Column", color="#3b82f6")
        plt.title("Missing Value Percentage by Column (Top 20)", fontsize=14, fontweight="bold")
        plt.xlabel("Missing Percentage (%)")
        plt.ylabel("Column Name")
        plt.tight_layout()
        plot_path = FIGURES_DIR / "missing_values_inspection.png"
        plt.savefig(plot_path, dpi=300)
        print(f"✓ Saved missing values plot to: {plot_path}")
        plt.show()
    else:
        print("✓ No missing values detected in the loaded sample!")
else:
    print("Awaiting dataset placement.")


Awaiting dataset placement.


### Step 7: Duplicate Rows and Date Range Verification
We verify whether duplicate flight records exist and inspect the temporal span of the dataset.

In [7]:
if not df.empty:
    num_duplicates = df.duplicated().sum()
    print(f"Total Duplicate Rows: {num_duplicates:,} ({(num_duplicates / len(df)) * 100:.3f}%)")
    
    # Date Range Detection
    detected_mapping, _ = detect_column_mapping(df.columns)
    if "date" in detected_mapping:
        dcol = detected_mapping["date"]
        dates = pd.to_datetime(df[dcol].dropna(), errors="coerce")
        print(f"Date Coverage: from {dates.min().strftime('%Y-%m-%d')} to {dates.max().strftime('%Y-%m-%d')}")
    elif {"year", "month", "day"}.issubset(detected_mapping.keys()):
        y = df[detected_mapping["year"]]
        m = df[detected_mapping["month"]]
        d = df[detected_mapping["day"]]
        dates = pd.to_datetime(pd.DataFrame({"year": y, "month": m, "day": d}).dropna(), errors="coerce")
        print(f"Date Coverage: from {dates.min().strftime('%Y-%m-%d')} to {dates.max().strftime('%Y-%m-%d')}")
    else:
        print("Date column not directly identified; check year/month/day columns.")
else:
    print("Awaiting dataset placement.")


Awaiting dataset placement.


### Step 8: Cancellation and Diversion Handling
In aviation analytics, **cancelled** and **diverted** flights represent operational anomalies:
- A cancelled flight never departed or arrived; its arrival delay is undefined (NaN).
- A diverted flight arrived at a different airport, altering scheduled arrival dynamics.

For **arrival delay prediction**, the standard methodology is to filter out cancelled and diverted flights before modeling, or treat cancellation as a distinct pre-flight classification task.

In [8]:
if not df.empty:
    detected_mapping, _ = detect_column_mapping(df.columns)
    
    if "cancellation" in detected_mapping:
        canc_col = detected_mapping["cancellation"]
        canc_count = (df[canc_col] == 1).sum()
        print(f"Cancelled Flights : {canc_count:,} ({canc_count / len(df) * 100:.2f}%)")
        
    if "diversion" in detected_mapping:
        div_col = detected_mapping["diversion"]
        div_count = (df[div_col] == 1).sum()
        print(f"Diverted Flights  : {div_count:,} ({div_count / len(df) * 100:.2f}%)")
else:
    print("Awaiting dataset placement.")


Awaiting dataset placement.


### Step 9: Target Variable Construction (`is_delayed_15`)
According to the Federal Aviation Administration (FAA) standard:
- A flight is considered **delayed** if its arrival delay is **15 minutes or greater**.
$$\text{Target} = \begin{cases} 1 & \text{if } \text{Arrival Delay} \ge 15 \text{ minutes} \\ 0 & \text{if } \text{Arrival Delay} < 15 \text{ minutes} \end{cases}$$

In [9]:
if not df.empty:
    detected_mapping, _ = detect_column_mapping(df.columns)
    
    if "arrival_delay" in detected_mapping:
        arr_col = detected_mapping["arrival_delay"]
        
        # Filter completed flights (non-cancelled, non-diverted) with valid arrival delays
        filter_mask = df[arr_col].notnull()
        if "cancellation" in detected_mapping:
            filter_mask &= (df[detected_mapping["cancellation"]] == 0)
        if "diversion" in detected_mapping:
            filter_mask &= (df[detected_mapping["diversion"]] == 0)
            
        completed_df = df[filter_mask].copy()
        completed_df[TARGET_COLUMN] = (completed_df[arr_col] >= DELAY_THRESHOLD_MINUTES).astype(int)
        
        counts = completed_df[TARGET_COLUMN].value_counts()
        pcts = completed_df[TARGET_COLUMN].value_counts(normalize=True) * 100
        
        print("=" * 60)
        print("TARGET DISTRIBUTION (Arrival Delay >= 15 min)")
        print("=" * 60)
        print(f"On-Time / Minor Delay (0) : {counts.get(0, 0):,} ({pcts.get(0, 0):.2f}%)")
        print(f"Delayed >= 15 min (1)      : {counts.get(1, 0):,} ({pcts.get(1, 0):.2f}%)")
        print(f"Total Completed Flights    : {len(completed_df):,}")
        print("=" * 60)
        
        # Visualize Target Distribution
        plt.figure(figsize=(7, 5))
        sns.barplot(x=["On-Time / Minor (<15m)", "Delayed (>=15m)"], y=counts.values, palette=["#10b981", "#ef4444"])
        plt.title("Target Class Distribution (Arrival Delay >= 15 min)", fontsize=13, fontweight="bold")
        plt.ylabel("Number of Flights")
        for i, val in enumerate(counts.values):
            plt.text(i, val + (len(completed_df) * 0.01), f"{val:,}\n({pcts.iloc[i]:.1f}%)", ha="center", fontweight="bold")
        plt.tight_layout()
        plot_path = FIGURES_DIR / "target_distribution.png"
        plt.savefig(plot_path, dpi=300)
        print(f"✓ Saved target distribution plot to: {plot_path}")
        plt.show()
    else:
        print("Arrival delay column not detected; target cannot be constructed.")
else:
    print("Awaiting dataset placement.")


Awaiting dataset placement.


## 📝 Summary of Findings & Next Steps (Phase 1)

### Key Takeaways from Data Inspection:
1. **Target Definition**: Binary classification where $y = 1$ when `Arrival Delay >= 15 minutes`.
2. **Leakage Prevention**: Strictly excluded all post-departure variables (`DepTime`, `DepDelay`, `ArrTime`, `TaxiOut`, `WheelsOff`, etc.) from model features.
3. **Operational Filtering**: Filtered out cancelled and diverted flights since their arrival delay is undefined.
4. **Class Imbalance**: Typical domestic flight data shows ~18-22% delay rate, requiring balanced evaluation metrics (PR-AUC, ROC-AUC, F1-score) rather than accuracy alone.

### Next Steps for Phase 1 (Traditional ML):
- **Feature Engineering**: Cyclic encoding for scheduled departure/arrival hours (sin/cos), route historical delay rates (leave-one-out/out-of-fold target encoding), airline encoding, airport density.
- **Model Training**: Baseline Logistic Regression, Random Forest, and Gradient Boosting (XGBoost).
- **Evaluation**: Time-based cross-validation (train on earlier months, test on later months to avoid future information leakage).
